# Reference data: countries + subnational locations

Build committed lookup tables under `data/reference/` used by later clean / merge notebooks (spec `04`):

| Output | Role |
|--------|------|
| `countries.csv` | ISO-2 master list, name aliases, VectorNet / Europe–Med flags |
| `subnational_locations.csv` | Curated admin / NUTS centroids for Tier 1 Path B |

**Sources (downloaded into `data/reference/.cache/`, not committed):**

- ISO 3166-1 via `pycountry`
- Eurostat GISCO NUTS 2021 label points (`NUTS_LB_2021_4326.geojson`)
- Natural Earth 10m admin-1 states/provinces

**Virus inputs (multi-source extract):** inventory country tokens and aliases from all feeds (`ena_sequence`, `ena_read_run`, `bvbrc`, `pathoplexus`, `empres_i`, `ecdc_atlas`), and resolve centroids from `Country: region` strings, dedicated `subnational_region`, and `nuts_id` (ECDC NUTS3). Also include VectorNet `higherGeographyID` NUTS codes.

**Rules:** match region / NUTS keys when the administrative scale is comparable to H3 res 4–5. Do **not** invent centroids for unmatched city / locality strings; leave those country-only until curated.


In [1]:
from __future__ import annotations

import json
import re
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd
import pycountry
import shapefile
from IPython.display import display

from vector_guard.data import (
    default_reference_dir,
    find_repo_root,
    load_vector_detections,
    load_virus_detections,
)

REPO_ROOT = find_repo_root(Path.cwd())
REF_DIR = default_reference_dir(REPO_ROOT)
CACHE_DIR = REF_DIR / ".cache"
REF_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

COUNTRIES_PATH = REF_DIR / "countries.csv"
SUBNATIONAL_PATH = REF_DIR / "subnational_locations.csv"

NUTS_LB_URL = (
    "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
    "NUTS_LB_2021_4326.geojson"
)
NUTS_LB_PATH = CACHE_DIR / "NUTS_LB_2021_4326.geojson"
NE_ADMIN1_URL = (
    "https://naciscdn.org/naturalearth/10m/cultural/"
    "ne_10m_admin_1_states_provinces.zip"
)
NE_ADMIN1_ZIP = CACHE_DIR / "ne_10m_admin_1_states_provinces.zip"
NE_ADMIN1_SHP = CACHE_DIR / "ne_10m_admin_1_states_provinces.shp"

DOWNLOAD_TIMEOUT_S = 300

REPO_ROOT, REF_DIR

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/reference'))

## Helpers

Normalization, downloads, and country / region alias tables (including multi-source EMPRES-i / ECDC / Pathoplexus / BV-BRC spellings).


In [2]:
def download_file(url: str, dest: Path, timeout_s: int = DOWNLOAD_TIMEOUT_S) -> Path:
    """Download *url* to *dest* if missing."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = Request(url, headers={"User-Agent": "vector-guard/0.1 (reference build)"})
    with urlopen(req, timeout=timeout_s) as resp, dest.open("wb") as out:
        while True:
            chunk = resp.read(1024 * 256)
            if not chunk:
                break
            out.write(chunk)
    return dest


def norm_key(text: str) -> str:
    """Casefold, strip accents/punctuation → hyphenated match key."""
    if not text:
        return ""
    s = unicodedata.normalize("NFKD", str(text))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.casefold().strip().replace("&", " and ")
    s = re.sub(r"[\'\`´]", "", s)
    s = re.sub(r"[^a-z0-9]+", "-", s)
    return re.sub(r"-+", "-", s).strip("-")


def strip_admin_suffix(text: str) -> str:
    return re.sub(
        r"\b(province|region|state|county|oblast|department|prefecture|republic)\b",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip(" ,/-")


# VectorNet publishes UK / EL; ISO uses GB / GR.
VECTORNET_CODE_TO_ISO2 = {"UK": "GB", "EL": "GR"}
ISO2_TO_VECTORNET_CODE = {"GB": "UK", "GR": "EL"}

# Extra / non-ISO rows we still want in countries.csv
EXTRA_COUNTRIES = [
    {
        "iso2": "XK",
        "name": "Kosovo",
        "name_aliases": "Kosovo|Kosovo, Republic of",
    },
    {
        # Historic / dissolved; still appears in older ENA country tokens
        "iso2": "AN",
        "name": "Netherlands Antilles",
        "name_aliases": "Netherlands Antilles",
    },
]

# Manual name → ISO-2 (covers ENA spellings pycountry misses)
COUNTRY_NAME_ALIASES: dict[str, str] = {
    "usa": "US",
    "united states": "US",
    "united states of america": "US",
    "uk": "GB",
    "united kingdom": "GB",
    "russia": "RU",
    "russian federation": "RU",
    "viet nam": "VN",
    "vietnam": "VN",
    "laos": "LA",
    "lao peoples democratic republic": "LA",
    "cote divoire": "CI",
    "ivory coast": "CI",
    "south korea": "KR",
    "korea republic of": "KR",
    "republic of korea": "KR",
    "north korea": "KP",
    "iran": "IR",
    "syria": "SY",
    "moldova": "MD",
    "tanzania": "TZ",
    "bolivia": "BO",
    "venezuela": "VE",
    "taiwan": "TW",
    "palestine": "PS",
    "west bank": "PS",
    "czech republic": "CZ",
    "czechia": "CZ",
    "turkey": "TR",
    "turkiye": "TR",
    "brunei": "BN",
    "cape verde": "CV",
    "east timor": "TL",
    "timor-leste": "TL",
    "reunion": "RE",
    "réunion": "RE",
    "french guiana": "GF",
    "french polynesia": "PF",
    "puerto rico": "PR",
    "kosovo": "XK",
    "north macedonia": "MK",
    "macedonia": "MK",
    "democratic republic of the congo": "CD",
    "republic of the congo": "CG",
    "congo": "CG",
    "swaziland": "SZ",
    "eswatini": "SZ",
    "myanmar": "MM",
    "burma": "MM",
    "bosnia and herzegovina": "BA",
    "micronesia": "FM",
    "curacao": "CW",
    "hong kong": "HK",
    "macao": "MO",
    "macau": "MO",
    "france martinique": "MQ",
    "france guadeloupe": "GP",
    "martinique": "MQ",
    "guadeloupe": "GP",
    "mayotte": "YT",
    "new caledonia": "NC",
    "wallis and futuna": "WF",
    "saint martin": "MF",
    "saint barthelemy": "BL",
    "british virgin islands": "VG",
    "virgin islands": "VI",
    "netherlands antilles": "AN",
    "ussr": "RU",
    "yugoslavia": "RS",
    "serbia and montenegro": "RS",
    "new delhi india": "IN",
    "thailand thailand": "TH",
    "vatican": "VA",
    "holy see": "VA",
    # Multi-source spellings (EMPRES-i / ENA / Pathoplexus / BV-BRC)
    "the united kingdom of great britain and northern ireland": "GB",
    "united kingdom of great britain and northern ireland": "GB",
    "netherlands kingdom of the": "NL",
    "china hong kong special administrative region": "HK",
    "hong kong special administrative region": "HK",
    "sint maarten": "SX",
    "sint maarten dutch part": "SX",
    "czechoslovakia": "CZ",
    "505 nicaragua": "NI",
    "korea": "KR",
}

# Europe / Mediterranean basin flag (from 04_eda_viruses working list → ISO-2)
EUROPE_MED_ISO2 = {
    "AL", "AD", "AT", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK", "EE", "FI",
    "FR", "DE", "GR", "HU", "IS", "IE", "IL", "IT", "XK", "LV", "LB", "LI", "LT",
    "LU", "MT", "MD", "MC", "ME", "MA", "NL", "MK", "NO", "PL", "PT", "RO", "RU",
    "SM", "RS", "SK", "SI", "ES", "SE", "CH", "SY", "TN", "TR", "UA", "GB",
    "DZ", "EG", "LY", "PS", "JO", "GE", "AM", "AZ", "EH",
}

# Curated region aliases: (country_iso2, alias_key) → (match_country_iso2, match_key)
# Used when ENA spellings diverge from NUTS / Natural Earth names.
REGION_ALIASES: dict[tuple[str, str], tuple[str, str]] = {
    ("IT", "sardinia"): ("IT", "sardegna"),
    ("IT", "sicily"): ("IT", "sicilia"),
    ("IT", "emiliaromagna"): ("IT", "emilia-romagna"),
    ("IT", "emilia-romagna"): ("IT", "emilia-romagna"),
    ("IT", "tuscany"): ("IT", "toscana"),
    ("IT", "lombardy"): ("IT", "lombardia"),
    ("IT", "piedmont"): ("IT", "piemonte"),
    ("IT", "apulia"): ("IT", "puglia"),
    ("FR", "corsica"): ("FR", "corse"),
    ("FR", "corse"): ("FR", "corse"),
    ("FR", "guyane"): ("FR", "guyane"),
    ("DE", "north-rhine-westphalia"): ("DE", "nordrhein-westfalen"),
    ("DE", "bavaria"): ("DE", "bayern"),
    ("DE", "lower-saxony"): ("DE", "niedersachsen"),
    ("DE", "saxony"): ("DE", "sachsen"),
    ("DE", "hesse"): ("DE", "hessen"),
    ("ES", "andalusia"): ("ES", "andalucia"),
    ("ES", "catalonia"): ("ES", "cataluna"),
    ("ES", "valencia"): ("ES", "comunitat-valenciana"),
    ("US", "puerto-rico"): ("PR", "puerto-rico"),
    ("CN", "guangdong-province"): ("CN", "guangdong"),
    ("CN", "yunnan-province"): ("CN", "yunnan"),
    ("AU", "nsw"): ("AU", "new-south-wales"),
    ("ZA", "transvaal"): ("ZA", "gauteng"),  # historical → approximate modern admin1
}

print("alias tables ready")


alias tables ready


## Load detection extracts

Requires raw copies from notebooks `01` and `03` (multi-source `virus_detections.tsv`).


In [3]:
vectors = load_vector_detections()
viruses = load_virus_detections()

vectors["countryCode"] = (
    vectors["higherGeographyID"].astype("string").str[:2].str.upper()
)
print(f"vectors: {len(vectors):,} rows; {vectors['countryCode'].nunique()} country codes")
print(f"viruses: {len(viruses):,} rows; sources={sorted(viruses['source'].dropna().unique())}")
print("virus source counts:")
display(viruses["source"].value_counts().rename("rows").to_frame())
print(
    "geo fields nonempty — "
    f"country={(viruses['country'].str.len() > 0).sum():,}, "
    f"subnational_region={(viruses['subnational_region'].str.len() > 0).sum():,}, "
    f"nuts_id={(viruses['nuts_id'].str.len() > 0).sum():,}"
)


vectors: 120,158 rows; 58 country codes
viruses: 168,277 rows; sources=['bvbrc', 'ecdc_atlas', 'empres_i', 'ena_read_run', 'ena_sequence', 'pathoplexus']
virus source counts:


,rows
source,
ena_sequence,118381
empres_i,25874
ena_read_run,13231
pathoplexus,6377
bvbrc,4108
ecdc_atlas,306


geo fields nonempty — country=166,945, subnational_region=86,892, nuts_id=306


## Download centroid reference caches

In [4]:
download_file(NUTS_LB_URL, NUTS_LB_PATH)
download_file(NE_ADMIN1_URL, NE_ADMIN1_ZIP)

if not NE_ADMIN1_SHP.exists():
    with zipfile.ZipFile(NE_ADMIN1_ZIP) as zf:
        zf.extractall(CACHE_DIR)

assert NUTS_LB_PATH.exists(), NUTS_LB_PATH
assert NE_ADMIN1_SHP.exists(), NE_ADMIN1_SHP
print("NUTS LB bytes:", NUTS_LB_PATH.stat().st_size)
print("NE admin1 shp:", NE_ADMIN1_SHP)

NUTS LB bytes: 1056468
NE admin1 shp: /workspaces/vector-guard/data/reference/.cache/ne_10m_admin_1_states_provinces.shp


## Build `countries.csv`

ISO-3166 base + observed multi-source virus country aliases + VectorNet / Europe–Med flags.


In [5]:
def build_name_to_iso2() -> dict[str, str]:
    mapping: dict[str, str] = {}
    for country in pycountry.countries:
        mapping[norm_key(country.name)] = country.alpha_2
        for attr in ("official_name", "common_name"):
            value = getattr(country, attr, None)
            if value:
                mapping[norm_key(value)] = country.alpha_2
    for alias, iso2 in COUNTRY_NAME_ALIASES.items():
        mapping[norm_key(alias)] = iso2
    for row in EXTRA_COUNTRIES:
        mapping[norm_key(row["name"])] = row["iso2"]
        for alias in row.get("name_aliases", "").split("|"):
            if alias.strip():
                mapping[norm_key(alias)] = row["iso2"]
    return mapping


NAME_TO_ISO2 = build_name_to_iso2()


def parse_country_token(value: str) -> str:
    text = (value or "").strip()
    if not text:
        return ""
    return text.split(":", 1)[0].strip()


virus_country_tokens = (
    viruses["country"].fillna("").map(parse_country_token)
)
virus_token_counts = Counter(t for t in virus_country_tokens if t)

vector_iso2 = {
    VECTORNET_CODE_TO_ISO2.get(code, code)
    for code in vectors["countryCode"].dropna().unique()
    if re.fullmatch(r"[A-Z]{2}", str(code))
}

# Drop obvious non-country prefixes seen in VectorNet (e.g. '3-', '6-')
vector_iso2 = {c for c in vector_iso2 if c.isalpha()}

rows: list[dict] = []
seen_iso2: set[str] = set()

for country in sorted(pycountry.countries, key=lambda c: c.alpha_2):
    aliases: set[str] = set()
    canonical = country.name
    # Prefer shorter common name when present (e.g. Taiwan)
    if hasattr(country, "common_name") and country.common_name:
        canonical = country.common_name
    for attr in ("name", "official_name", "common_name"):
        value = getattr(country, attr, None)
        if value and norm_key(value) != norm_key(canonical):
            aliases.add(value)
    seen_iso2.add(country.alpha_2)
    rows.append(
        {
            "iso2": country.alpha_2,
            "name": canonical,
            "name_aliases": aliases,
            "in_vectornet_scope": 0,
            "europe_med": 0,
        }
    )

for extra in EXTRA_COUNTRIES:
    if extra["iso2"] in seen_iso2:
        continue
    rows.append(
        {
            "iso2": extra["iso2"],
            "name": extra["name"],
            "name_aliases": {
                a for a in extra.get("name_aliases", "").split("|") if a.strip()
            },
            "in_vectornet_scope": 0,
            "europe_med": 0,
        }
    )
    seen_iso2.add(extra["iso2"])

by_iso2 = {r["iso2"]: r for r in rows}

# Attach virus spelling aliases observed in the extract
unmapped_tokens: list[tuple[str, int]] = []
for token, count in virus_token_counts.items():
    iso2 = NAME_TO_ISO2.get(norm_key(token))
    if not iso2 or iso2 not in by_iso2:
        unmapped_tokens.append((token, count))
        continue
    # Keep observed spellings even when they only differ by accents/punctuation
    if token != by_iso2[iso2]["name"]:
        by_iso2[iso2]["name_aliases"].add(token)

# Flags
for iso2, row in by_iso2.items():
    row["europe_med"] = 1 if iso2 in EUROPE_MED_ISO2 else 0
    row["in_vectornet_scope"] = 1 if (iso2 in vector_iso2 or iso2 in EUROPE_MED_ISO2) else 0

countries = pd.DataFrame(
    [
        {
            "iso2": r["iso2"],
            "name": r["name"],
            "name_aliases": "|".join(sorted(a for a in r["name_aliases"] if a)),
            "in_vectornet_scope": r["in_vectornet_scope"],
            "europe_med": r["europe_med"],
        }
        for r in sorted(by_iso2.values(), key=lambda x: x["iso2"])
    ]
)
countries["name_aliases"] = countries["name_aliases"].fillna("")

countries.to_csv(COUNTRIES_PATH, index=False)
print(f"Wrote {COUNTRIES_PATH} ({len(countries):,} rows)")
print(
    "in_vectornet_scope=",
    int(countries["in_vectornet_scope"].sum()),
    "europe_med=",
    int(countries["europe_med"].sum()),
    "with_aliases=",
    int((countries["name_aliases"] != "").sum()),
)
print(f"Unmapped virus country tokens: {len(unmapped_tokens)}")
unmapped_df = (
    pd.DataFrame(unmapped_tokens, columns=["token", "rows"])
    .sort_values("rows", ascending=False)
)
display(unmapped_df.head(25))

# Per-source view of remaining unmapped tokens (should be oceans / missing / non-countries)
unmapped_set = {t for t, _ in unmapped_tokens}
per_source_unmapped = []
for source, group in viruses.groupby("source", dropna=False):
    tokens = group["country"].fillna("").map(parse_country_token)
    counts = Counter(t for t in tokens if t and t in unmapped_set)
    for token, n in counts.items():
        per_source_unmapped.append({"source": source, "token": token, "rows": n})
if per_source_unmapped:
    print("Unmapped tokens by source (non-countries / ambiguous leftovers):")
    display(
        pd.DataFrame(per_source_unmapped)
        .sort_values(["source", "rows"], ascending=[True, False])
    )
else:
    print("No unmapped country tokens.")
display(countries.head(10))


Wrote /workspaces/vector-guard/data/reference/countries.csv (251 rows)
in_vectornet_scope= 60 europe_med= 60 with_aliases= 181
Unmapped virus country tokens: 9


,token,rows
0,missing,202
6,Not Applicable,182
5,Not applicable,89
1,Pacific Ocean,14
3,Borneo,7
2,Atlantic Ocean,2
4,Not collected,1
7,Not provided,1
8,not collected,1


Unmapped tokens by source (non-countries / ambiguous leftovers):


,source,token,rows
0,bvbrc,not collected,1
5,ena_read_run,Not Applicable,182
1,ena_read_run,missing,157
4,ena_read_run,Not applicable,89
2,ena_read_run,Borneo,2
3,ena_read_run,Not collected,1
6,ena_read_run,Not provided,1
8,ena_sequence,Pacific Ocean,14
7,ena_sequence,missing,6
10,ena_sequence,Borneo,5


,iso2,name,name_aliases,in_vectornet_scope,europe_med
0,AD,Andorra,Principality of Andorra,1,1
1,AE,United Arab Emirates,,0,0
2,AF,Afghanistan,Islamic Republic of Afghanistan,0,0
3,AG,Antigua and Barbuda,,0,0
4,AI,Anguilla,,0,0
5,AL,Albania,Republic of Albania,1,1
6,AM,Armenia,Republic of Armenia,1,1
7,AN,Netherlands Antilles,Netherlands Antilles,0,0
8,AO,Angola,Republic of Angola,0,0
9,AQ,Antarctica,,0,0


## Build centroid indexes (NUTS + Natural Earth)

In [6]:
nuts_geo = json.loads(NUTS_LB_PATH.read_text(encoding="utf-8"))

nuts_by_id: dict[str, dict] = {}
nuts_by_key: dict[tuple[str, str], list[dict]] = defaultdict(list)

for feat in nuts_geo["features"]:
    props = feat["properties"]
    nuts_id = str(props["NUTS_ID"]).upper()
    cntr = str(props["CNTR_CODE"]).upper()
    iso2 = VECTORNET_CODE_TO_ISO2.get(cntr, cntr)
    level = int(props["LEVL_CODE"])
    lat = float(props["LAT"])
    lon = float(props["LON"])
    name = props.get("NUTS_NAME") or props.get("NAME_LATN") or nuts_id
    entry = {
        "country_iso2": iso2,
        "region_key": norm_key(nuts_id),
        "region_type": {0: "admin1", 1: "admin1", 2: "admin1", 3: "nuts3"}[level]
        if level != 3
        else "nuts3",
        "canonical_name": name,
        "lat": lat,
        "lon": lon,
        "source_authority": "NUTS_2021",
        "nuts_id": nuts_id,
        "nuts_level": level,
        "cntr_code": cntr,
    }
    # Prefer admin1-scale labels for name matching: L1/L2 as admin1; L3 as nuts3
    if level in (1, 2):
        entry["region_type"] = "admin1"
    elif level == 3:
        entry["region_type"] = "nuts3"
    else:
        entry["region_type"] = "admin1"

    nuts_by_id[nuts_id] = entry
    for label in {props.get("NUTS_NAME"), props.get("NAME_LATN")}:
        if not label:
            continue
        key = norm_key(label)
        nuts_by_key[(iso2, key)].append(entry)
        nuts_by_key[(cntr, key)].append(entry)
        key2 = norm_key(strip_admin_suffix(label))
        if key2 and key2 != key:
            nuts_by_key[(iso2, key2)].append(entry)
            nuts_by_key[(cntr, key2)].append(entry)

ne_by_key: dict[tuple[str, str], list[dict]] = defaultdict(list)
sf = shapefile.Reader(str(NE_ADMIN1_SHP))
for rec in sf.records():
    d = rec.as_dict()
    iso2 = (d.get("iso_a2") or "").upper()
    if not iso2 or iso2 in {"-99", "XX"}:
        continue
    lat, lon = d.get("latitude"), d.get("longitude")
    if lat in (None, "") or lon in (None, ""):
        continue
    name = d.get("name") or d.get("name_en") or ""
    if not name:
        continue
    type_en = (d.get("type_en") or "").lower()
    region_type = "admin2" if "province" in type_en or "county" in type_en else "admin1"
    entry = {
        "country_iso2": iso2,
        "region_key": norm_key(name),
        "region_type": region_type,
        "canonical_name": name,
        "lat": float(lat),
        "lon": float(lon),
        "source_authority": "NaturalEarth_10m_admin1",
    }
    labels = [
        d.get("name"),
        d.get("name_en"),
        d.get("gn_name"),
        d.get("woe_name"),
        d.get("abbrev"),
        d.get("postal"),
    ]
    alt = d.get("name_alt") or ""
    labels.extend(part.strip() for part in alt.split("|") if part.strip())
    for label in labels:
        if not label:
            continue
        key = norm_key(str(label))
        ne_by_key[(iso2, key)].append(entry)
        key2 = norm_key(strip_admin_suffix(str(label)))
        if key2 and key2 != key:
            ne_by_key[(iso2, key2)].append(entry)

print(f"NUTS entries: {len(nuts_by_id):,}; name keys: {len(nuts_by_key):,}")
print(f"Natural Earth name keys: {len(ne_by_key):,}")

NUTS entries: 2,010; name keys: 2,257
Natural Earth name keys: 16,105


## Inventory virus regions and resolve centroids

Collect distinct region keys from:

1. ENA-style `country` values `Country: region`
2. Dedicated `subnational_region` (Pathoplexus division, BV-BRC state/province, EMPRES-i locality/region, sample backfill)
3. `nuts_id` (ECDC NUTS3 / GAUL codes)

Resolve each `(country_iso2, region)` against NUTS 2021 label points and Natural Earth admin-1. Unmatched free-text localities stay out of the CSV (country-only Tier 2).


In [7]:
def split_country_region(value: str) -> tuple[str, str]:
    text = (value or "").strip()
    if not text:
        return "", ""
    if ":" in text:
        country, region = text.split(":", 1)
        return country.strip(), region.strip()
    return text, ""


def region_candidate_keys(region: str) -> list[str]:
    """Ordered normalized keys to try for a free-text region string."""
    raw_parts = [region]
    # Comma segments (with or without spaces): City,State / City, Province
    if "," in region:
        parts = [p.strip() for p in region.split(",") if p.strip()]
        raw_parts.extend(parts)
        if len(parts) >= 2:
            raw_parts.append(parts[-1])  # prefer trailing admin unit
            raw_parts.append(parts[0])
    if " - " in region:
        raw_parts.append(region.split(" - ", 1)[0].strip())
        raw_parts.append(region.split(" - ", 1)[-1].strip())
    cleaned = []
    for part in raw_parts:
        part = re.sub(r"\([^)]*\)", "", part).strip()
        if part:
            cleaned.append(part)
    keys: list[str] = []
    for part in cleaned:
        for variant in (part, strip_admin_suffix(part)):
            key = norm_key(variant)
            if key and key not in keys:
                keys.append(key)
    return keys


def pick_nuts(entries: list[dict]) -> dict:
    # Prefer coarser NUTS (L1 then L2 then L3) for H3 res 4–5 scale
    return sorted(entries, key=lambda e: e.get("nuts_level", 99))[0]


def resolve_region(country_iso2: str, region: str) -> tuple[dict | None, str]:
    keys = region_candidate_keys(region)
    # Apply curated aliases first
    for key in list(keys):
        alias = REGION_ALIASES.get((country_iso2, key))
        if alias:
            country_iso2, key = alias
            if key not in keys:
                keys.insert(0, key)
    for key in keys:
        # Exact NUTS id (e.g. ITI43)
        nuts_id = key.upper().replace("-", "")
        if re.fullmatch(r"[A-Z]{2}[A-Z0-9]{1,3}", nuts_id) and nuts_id in nuts_by_id:
            entry = dict(nuts_by_id[nuts_id])
            entry["region_key"] = norm_key(nuts_id)
            return entry, "nuts_id"
        for code in {country_iso2, ISO2_TO_VECTORNET_CODE.get(country_iso2, country_iso2)}:
            if (code, key) in nuts_by_key:
                entry = dict(pick_nuts(nuts_by_key[(code, key)]))
                entry["region_key"] = key
                entry["country_iso2"] = VECTORNET_CODE_TO_ISO2.get(
                    entry["country_iso2"], entry["country_iso2"]
                )
                return entry, f"nuts_l{entry['nuts_level']}"
        if (country_iso2, key) in ne_by_key:
            entry = dict(ne_by_key[(country_iso2, key)][0])
            entry["region_key"] = key
            return entry, "ne_admin1"
    return None, ""


def inventory_virus_regions(frame: pd.DataFrame) -> Counter:
    """Count (country_token, region) from country / subnational_region / nuts_id."""
    counts: Counter = Counter()
    sources: Counter = Counter()

    for value in frame["country"].fillna(""):
        country_token, region = split_country_region(value)
        if region:
            counts[(country_token, region)] += 1
            sources["country_embedded"] += 1

    for country_val, region in zip(
        frame["country"].fillna(""),
        frame["subnational_region"].fillna(""),
        strict=True,
    ):
        region = str(region).strip()
        if not region:
            continue
        country_token, embedded = split_country_region(country_val)
        # Avoid double-counting the same row when country already embeds the region.
        if embedded and norm_key(embedded) == norm_key(region):
            continue
        counts[(country_token, region)] += 1
        sources["subnational_region"] += 1

    for country_val, nuts_id in zip(
        frame["country"].fillna(""),
        frame["nuts_id"].fillna(""),
        strict=True,
    ):
        nuts_id = str(nuts_id).strip()
        if not nuts_id:
            continue
        country_token, _ = split_country_region(country_val)
        counts[(country_token, nuts_id)] += 1
        sources["nuts_id"] += 1

    print("Inventory contributions (row events):", dict(sources))
    return counts


region_counts = inventory_virus_regions(viruses)

resolved_rows: list[dict] = []
resolve_stats = Counter()
unresolved: list[tuple[str, str, int]] = []
seen_keys: set[tuple[str, str]] = set()

for (country_token, region), n in region_counts.items():
    iso2 = NAME_TO_ISO2.get(norm_key(country_token))
    if not iso2:
        resolve_stats["no_country"] += n
        unresolved.append((country_token, region, n))
        continue
    entry, source = resolve_region(iso2, region)
    if entry is None:
        resolve_stats["unresolved"] += n
        unresolved.append((country_token, region, n))
        continue
    key = (entry["country_iso2"], entry["region_key"])
    resolve_stats[source] += n
    if key in seen_keys:
        continue
    seen_keys.add(key)
    resolved_rows.append(
        {
            "country_iso2": entry["country_iso2"],
            "region_key": entry["region_key"],
            "region_type": entry["region_type"],
            "canonical_name": entry["canonical_name"],
            "lat": round(float(entry["lat"]), 6),
            "lon": round(float(entry["lon"]), 6),
            "source_authority": entry["source_authority"],
        }
    )

# Alias keys as extra join rows (same centroid) for virus-facing spellings
for (alias_iso, alias_key), (match_iso2, match_key) in REGION_ALIASES.items():
    hit = None
    source_authority = ""
    vn_code = ISO2_TO_VECTORNET_CODE.get(match_iso2, match_iso2)
    if (match_iso2, match_key) in nuts_by_key:
        hit = pick_nuts(nuts_by_key[(match_iso2, match_key)])
        source_authority = "NUTS_2021"
    elif (vn_code, match_key) in nuts_by_key:
        hit = pick_nuts(nuts_by_key[(vn_code, match_key)])
        source_authority = "NUTS_2021"
    elif (match_iso2, match_key) in ne_by_key:
        hit = ne_by_key[(match_iso2, match_key)][0]
        source_authority = "NaturalEarth_10m_admin1"
    if hit is None:
        continue
    key = (alias_iso, alias_key)
    if key in seen_keys:
        continue
    seen_keys.add(key)
    resolved_rows.append(
        {
            "country_iso2": alias_iso,
            "region_key": alias_key,
            "region_type": hit.get("region_type", "admin1"),
            "canonical_name": hit.get("canonical_name", alias_key),
            "lat": round(float(hit["lat"]), 6),
            "lon": round(float(hit["lon"]), 6),
            "source_authority": source_authority,
        }
    )

print("Virus region row-events:", sum(region_counts.values()))
print("Distinct (country, region) keys:", len(region_counts))
print("Resolve stats (row-weighted):", dict(resolve_stats))
print("Distinct resolved keys so far:", len(resolved_rows))
print("Top unresolved:")
display(
    pd.DataFrame(unresolved, columns=["country", "region", "rows"])
    .sort_values("rows", ascending=False)
    .head(25)
)


Inventory contributions (row events): {'country_embedded': 46266, 'subnational_region': 40626, 'nuts_id': 306}


Virus region row-events: 87198
Distinct (country, region) keys: 28401
Resolve stats (row-weighted): {'ne_admin1': 41229, 'unresolved': 39141, 'nuts_l2': 3594, 'nuts_l3': 1977, 'nuts_l1': 889, 'no_country': 65, 'nuts_l0': 16, 'nuts_id': 287}
Distinct resolved keys so far: 1826
Top unresolved:


,country,region,rows
426,Viet Nam,South,1216
445,China,Guangzhou,714
543,Viet Nam,Nha Trang,562
458,China,Xi Shuang Ban Na,259
424,Viet Nam,south,240
563,Peru,Iquitos,211
459,China,Hangzhou,171
460,China,(Guangzhou),141
501,Kenya,Mombasa,133
449,Indonesia,Surabaya,120


## Add VectorNet NUTS `higherGeographyID` centroids

Exact NUTS codes from VectorNet become `region_type=nuts3` (or admin1 for L1/L2) rows. Virus `nuts_id` values were already inventoried above. GAUL-style IDs (e.g. `ILG11614`) are skipped — no invented centroids.


In [8]:
vector_ids = (
    vectors["higherGeographyID"].astype("string").str.upper().dropna().unique()
)
vn_added = 0
vn_missing = Counter()
for nuts_id in vector_ids:
    if nuts_id not in nuts_by_id:
        vn_missing[nuts_id] += 1
        continue
    entry = nuts_by_id[nuts_id]
    key = (entry["country_iso2"], norm_key(nuts_id))
    if key in seen_keys:
        continue
    seen_keys.add(key)
    resolved_rows.append(
        {
            "country_iso2": entry["country_iso2"],
            "region_key": norm_key(nuts_id),
            "region_type": entry["region_type"],
            "canonical_name": entry["canonical_name"],
            "lat": round(float(entry["lat"]), 6),
            "lon": round(float(entry["lon"]), 6),
            "source_authority": "NUTS_2021",
        }
    )
    vn_added += 1

print(f"VectorNet NUTS rows added: {vn_added}")
print(
    f"VectorNet IDs not in NUTS 2021 (e.g. GAUL): "
    f"{sum(vn_missing.values())} distinct — left unmatched"
)
display(pd.Series(vn_missing).sort_values(ascending=False).head(15).to_frame("count"))

VectorNet NUTS rows added: 646
VectorNet IDs not in NUTS 2021 (e.g. GAUL): 358 distinct — left unmatched


,count
DZG1363,1
DZG1350,1
RUG12535,1
SEG226077,1
SEG226079,1
DZG1356,1
EGG265350,1
RUG12557,1
RUG12498,1
RUG12524,1


## Write `subnational_locations.csv`

In [9]:
subnational = (
    pd.DataFrame(resolved_rows)
    .drop_duplicates(subset=["country_iso2", "region_key"], keep="first")
    .sort_values(["country_iso2", "region_key"])
    .reset_index(drop=True)
)

required_cols = [
    "country_iso2",
    "region_key",
    "region_type",
    "canonical_name",
    "lat",
    "lon",
    "source_authority",
]
subnational = subnational[required_cols]
assert not subnational.duplicated(["country_iso2", "region_key"]).any()
assert subnational["lat"].between(-90, 90).all()
assert subnational["lon"].between(-180, 180).all()

subnational.to_csv(SUBNATIONAL_PATH, index=False)
print(f"Wrote {SUBNATIONAL_PATH} ({len(subnational):,} rows)")
print(subnational["region_type"].value_counts().to_dict())
print(subnational["source_authority"].value_counts().to_dict())
display(subnational.head(15))
display(subnational.tail(10))

Wrote /workspaces/vector-guard/data/reference/subnational_locations.csv (2,472 rows)
{'nuts3': 1312, 'admin1': 852, 'admin2': 308}
{'NUTS_2021': 1440, 'NaturalEarth_10m_admin1': 1032}


,country_iso2,region_key,region_type,canonical_name,lat,lon,source_authority
0,AE,abu-dhabi,admin1,Abu Dhabi,23.523800,54.081700,NaturalEarth_10m_admin1
1,AL,al011,nuts3,Dibër,41.603923,20.256142,NUTS_2021
2,AL,al012,nuts3,Durrës,41.493877,19.713075,NUTS_2021
3,AL,al013,nuts3,Kukës,42.065074,20.403464,NUTS_2021
4,AL,al014,nuts3,Lezhë,41.783803,19.847841,NUTS_2021
5,AL,al015,nuts3,Shkodër,42.163029,19.702093,NUTS_2021
6,AL,al021,nuts3,Elbasan,41.067845,20.197947,NUTS_2021
7,AL,al022,nuts3,Tiranë,41.264599,19.798793,NUTS_2021
8,AL,al031,nuts3,Berat,40.624321,20.121132,NUTS_2021
9,AL,al032,nuts3,Fier,40.764165,19.618887,NUTS_2021


,country_iso2,region_key,region_type,canonical_name,lat,lon,source_authority
2462,ZA,free-state,admin2,Free State,-28.5815,26.4914,NaturalEarth_10m_admin1
2463,ZA,gauteng,admin2,Gauteng,-26.1682,28.2074,NaturalEarth_10m_admin1
2464,ZA,kwazulu-natal,admin2,KwaZulu-Natal,-28.7468,30.8231,NaturalEarth_10m_admin1
2465,ZA,limpopo,admin2,Limpopo,-23.5859,29.1301,NaturalEarth_10m_admin1
2466,ZA,mpumalanga,admin2,Mpumalanga,-25.9893,30.1421,NaturalEarth_10m_admin1
2467,ZA,north-west,admin2,North West,-26.3999,25.7403,NaturalEarth_10m_admin1
2468,ZA,northern-cape,admin2,Northern Cape,-29.7437,20.9961,NaturalEarth_10m_admin1
2469,ZA,transvaal,admin2,Gauteng,-26.1682,28.2074,NaturalEarth_10m_admin1
2470,ZA,western-cape,admin2,Western Cape,-33.5035,20.9745,NaturalEarth_10m_admin1
2471,ZW,manicaland,admin2,Manicaland,-18.9399,32.1206,NaturalEarth_10m_admin1


## Coverage summary


In [10]:
n_region_rows = sum(region_counts.values())
matched_rows = n_region_rows - resolve_stats["unresolved"] - resolve_stats["no_country"]
print("=== countries.csv ===")
print(f"path: {COUNTRIES_PATH}")
print(f"rows: {len(countries):,}")
print(f"aliases non-empty: {(countries['name_aliases'] != '').sum():,}")
print(f"in_vectornet_scope: {countries['in_vectornet_scope'].sum():,}")
print(f"europe_med: {countries['europe_med'].sum():,}")
print(f"unmapped country tokens: {len(unmapped_tokens)}")
print()
print("=== subnational_locations.csv ===")
print(f"path: {SUBNATIONAL_PATH}")
print(f"rows: {len(subnational):,}")
print(
    f"virus region row-event coverage: {matched_rows:,} / {n_region_rows:,} "
    f"({100 * matched_rows / max(n_region_rows, 1):.1f}%)"
)
print("Resolve breakdown:", dict(resolve_stats))
print("Unresolved locality/free-text strings remain country-only until curated.")

# Joinability: virus rows with at least one resolvable subnational / nuts key
def row_has_resolvable_region(country: str, subnational: str, nuts_id: str) -> bool:
    country_token, embedded = split_country_region(country)
    iso2 = NAME_TO_ISO2.get(norm_key(country_token))
    if not iso2:
        return False
    candidates = [c for c in (embedded, subnational.strip(), nuts_id.strip()) if c]
    for cand in candidates:
        entry, _src = resolve_region(iso2, cand)
        if entry is not None:
            return True
    return False


join_flags = [
    row_has_resolvable_region(c, s, n)
    for c, s, n in zip(
        viruses["country"].fillna(""),
        viruses["subnational_region"].fillna(""),
        viruses["nuts_id"].fillna(""),
        strict=True,
    )
]
join_ready = int(sum(join_flags))
print(
    f"Virus rows with ≥1 resolvable subnational/nuts key: "
    f"{join_ready:,} / {len(viruses):,} ({100 * join_ready / len(viruses):.1f}%)"
)


=== countries.csv ===
path: /workspaces/vector-guard/data/reference/countries.csv
rows: 251
aliases non-empty: 181
in_vectornet_scope: 60
europe_med: 60
unmapped country tokens: 9

=== subnational_locations.csv ===
path: /workspaces/vector-guard/data/reference/subnational_locations.csv
rows: 2,472
virus region row-event coverage: 47,992 / 87,198 (55.0%)
Resolve breakdown: {'ne_admin1': 41229, 'unresolved': 39141, 'nuts_l2': 3594, 'nuts_l3': 1977, 'nuts_l1': 889, 'no_country': 65, 'nuts_l0': 16, 'nuts_id': 287}
Unresolved locality/free-text strings remain country-only until curated.


Virus rows with ≥1 resolvable subnational/nuts key: 47,705 / 168,277 (28.3%)
